# Impedance ML Engineering — Session 03
## Engineer interpretable spatial matrix features
**Core: 60 minutes · Classical ML · Optional extension: 20–40 minutes**

## Why this matters
Which information is lost by a diagonal mean, and can engineered features improve prediction?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Derive spatial features with explicit masks and coverage.
- Separate deterministic current-visit transforms from learned preprocessing.
- Compare feature sets on the same patient partition.
- Identify unstable ratios and missingness shortcuts.

## R-to-Python / prerequisite bridge
`apply` over matrices → explicit axis reductions; `rowMeans(...,na.rm=TRUE)` → `np.nanmean`; never confuse axis 0 (samples) with contact axes.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Feature engineering maps the measurement into hypotheses a model can use: level, spatial spread, distance decay, directionality, and coverage. An engineered feature should specify what information it discards and what its units mean. Deterministic within-visit calculations do not learn across patients; imputation, scaling, PCA, and target-guided selection do.

`np.nanmean(values,axis=...)` ignores missing values; all-missing regions need an explicit missing result and coverage count. `np.nanstd(...,ddof=1)` estimates a sample SD. Broadcasting a spatial mask across samples is valid only when axes match. A flattened feature order is part of the input schema, not a disposable implementation detail.

## Task 1 — Inspect aligned arrays (5 min)
Use the later-lesson input scaffold below. Predict `matrices.shape`, `mask.shape`, and y length; verify (264, 12, 12), the same boolean mask shape, and 264 targets. Inspect one value against the raw CSV join. Check each matrix follows its row’s current `visit_id`, not `target_visit_id`.

## Task 2 — Create distance masks (6 min)
Construct 12×12 stimulation/recording index grids with `np.meshgrid(...,indexing="ij")`. Build diagonal, near (distance 1), middle (2–5), and far (≥6) masks. Predict each mask’s entry count before execution. Include all directed off-diagonal entries. Hint: broadcast a 12×12 mask over the sample dimension.

## Task 3 — Calculate summaries and coverage (8 min)
For each visit derive diagonal mean/SD, near/middle/far means, and valid counts for each region. Use NaN-aware reductions; diagonal SD uses ddof=1. Reconcile diagonal/near/far means and n_diag_valid against `pairs` with `np.allclose(...,equal_nan=True,atol=1e-6)`. A missing region remains missing, not zero.

## Task 4 — Add shape-sensitive features (8 min)
Derive mean signed and mean absolute asymmetry using pairs valid in both M and M.T, excluding diagonal. Derive SD of the 12 valid row means and a near-minus-far contrast. Before coding, describe which feature can distinguish matrices with equal diagonal means. Hint: boolean masks must exclude unmatched transpose cells; the signed mean across both directions can cancel by construction, so inspect a defined upper-directed half for the signed feature.

## Task 5 — Audit scale, stability, and shortcuts (6 min)
Plot feature distributions for training rows only. Inspect low-coverage visits and identify a feature sensitive to missingness. Audit the suggestion `near_mean/far_mean`: define behavior when the denominator is zero or near zero; prefer a contrast if no stable ratio contract is justified. Do not tune clipping bounds on validation or test.

## Task 6 — Build feature-set pipelines (8 min)
Create three inputs: original 11 columns, original plus selected spatial summaries, and all 144 flattened current cells plus current metadata. Use a separate imputer/scaler inside each ridge pipeline. Keep identical train/validation indices and alpha=1 for this exploratory comparison. Predict how flattening changes feature count. Record feature names/order; do not fit global PCA here.

## Task 7 — Compare predictive value (7 min)
Report validation pair- and patient-average MAE, baseline differences, and transformed feature counts for each representation. Plot residuals by horizon. State whether extra spatial information helped under this split, without requiring improvement. Explain why more input columns can be harmful with only 54 training patients.

## Task 8 — Write the feature specification (4 min)
Describe every retained feature’s units, input grain, QC handling, available-at-prediction-time status, and missing-region behavior. Save the table under `artifacts/03/`. Propose an ablation that isolates one scientific hypothesis rather than trying every combination.

## Retrieval practice
Rebuild a far-region summary and its valid count for three matrices without reusing your previous expression. Explain why the count belongs beside the mean.

## Professional reasoning
1. Why is fitting a feature selector on all outcomes different from a deterministic within-visit feature?
2. What could a coverage feature learn about measurement protocol rather than physiology?
3. Why does a directed matrix not necessarily support symmetry-based compression?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Write one pure spatial-feature function and check invariants on a small invented asymmetric matrix: a transpose flips the upper-directed signed asymmetry, while the absolute asymmetry stays the same.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)